# WORK9 — 05 Rolling-Origin Backtest V0.1 — GPU

**Purpose:** test the Pair forecasting architecture across 12 consecutive monthly origins instead of one validation snapshot.

Default rolling origins:
`2025-01 → 2025-12`, each forecasting H1/H2/H3.

Key anti-leakage rule:
- FIT = older known labels only.
- CALIBRATION = last 3 known target months ending at each origin.
- Final model refit = FIT + CALIBRATION after iteration/threshold/gate selection.
- EVALUATION = H1/H2/H3 after the origin.
- Evaluation actuals are never used for fit, early stopping, Hurdle threshold, or behavior-gate selection.

Models evaluated:
- Naive-1, SeasonalNaive-12, MovingAverage-3, Croston-SBA, TSB
- LightGBM Tweedie
- Hurdle
- Behavior-gated hybrid

CatBoost is intentionally skipped in rolling V0.1 after poor corrected V0.2 validation quality.

**Frozen test remains locked. Targets from Apr-2026 onward are forbidden.**


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip -q install -U lightgbm pyarrow pyyaml scikit-learn pytest

import os, sys, json, hashlib
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

WORK9 = Path('/content/drive/MyDrive/work9')
CONFIG = WORK9 / '01_config'
MODELING = WORK9 / '02_src' / 'modeling'
FEATURES = WORK9 / '02_src' / 'features'
REPORT_ROOT = WORK9 / '06_reports' / 'rolling_backtest'
RUN_ROOT = WORK9 / '08_runs'
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

print('WORK9:', WORK9)
print('GPU check:')
os.system('nvidia-smi -L || true')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 142.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.5/386.5 kB 37.7 MB/s eta 0:00:00
WORK9: /content/drive/MyDrive/work9
GPU check:


0

## 1. Verify locked lineage

This notebook does not rerun Dataset, Feature Selection, or Model V0.2. It requires the accepted V0.2 candidate pointer and rebuilds only a dedicated origin-safe rolling feature panel in memory.


In [3]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


dataset_ptr = json.loads((CONFIG / 'current_dataset_run.json').read_text())
feature_ptr = json.loads((CONFIG / 'current_feature_run.json').read_text())
fs_ptr = json.loads((CONFIG / 'current_feature_selection_run.json').read_text())
model_ptr = json.loads((CONFIG / 'current_model_candidate_run.json').read_text())

assert dataset_ptr['status'] == 'PASS'
assert dataset_ptr['dataset_version'] == 'dataset_v012'
assert feature_ptr['status'] == 'PASS'
assert feature_ptr['pair_feature_version'] == 'pair_feature_v013'
assert fs_ptr['status'] == 'PASS'
assert fs_ptr['selection_version'] == 'feature_selection_v04'
assert model_ptr['status'] == 'PASS'
assert model_ptr['model_version'] == 'pair_modeling_v02', model_ptr
assert model_ptr['source_feature_run_id'] == feature_ptr['run_id']
assert model_ptr['source_feature_selection_run_id'] == fs_ptr['run_id']

PAIR_PANEL = Path(dataset_ptr['pair_panel_path'])
CANONICAL_PAIR_FEATURE = Path(feature_ptr['pair_feature_panel_path'])
SELECTED = Path(fs_ptr['pair_selected_path'])
MODEL_CONTRACT = CONFIG / 'model_contract_v02.yaml'
ROLLING_CONTRACT = CONFIG / 'rolling_backtest_contract_v01.yaml'

for p in [PAIR_PANEL, CANONICAL_PAIR_FEATURE, SELECTED, MODEL_CONTRACT, ROLLING_CONTRACT]:
    assert p.exists(), p

assert sha256_file(CANONICAL_PAIR_FEATURE) == feature_ptr['output_sha256']['pair_feature_panel']

print('Dataset:', dataset_ptr['run_id'])
print('Feature:', feature_ptr['run_id'])
print('Feature Selection:', fs_ptr['run_id'])
print('Accepted Model V0.2:', model_ptr['run_id'])
print('Canonical pair feature SHA256: PASS')


Dataset: core_dataset_v012_20260815T122509Z
Feature: feature_stage_v013_20260815T123431Z
Feature Selection: feature_selection_v04_20260815T130048Z
Accepted Model V0.2: pair_modeling_v02_20260815T130724Z
Canonical pair feature SHA256: PASS


## 2. Static/unit tests

These tests verify rolling-origin boundaries, label-honest split logic, intermittent baseline state, calibration-only gate selection, and stability summaries before GPU training starts.


In [4]:
!PYTHONPATH="/content/drive/MyDrive/work9/02_src/modeling:/content/drive/MyDrive/work9/02_src/features" \
 python -m pytest -q /content/drive/MyDrive/work9/07_tests/test_rolling_backtest_runner_v01.py


.......                                                                  [100%]
7 passed in 2.50s


## 3. Run 12-origin backtest

This is intentionally heavier than Notebook 04 because it retrains at every monthly origin. Do not interrupt the runtime while an origin is training.


In [5]:
for p in [MODELING, FEATURES]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from rolling_backtest_runner_v01 import run_rolling_backtest

run_id = 'rolling_backtest_v01_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
report_dir = REPORT_ROOT / run_id
run_dir = RUN_ROOT / run_id
report_dir.mkdir(parents=True, exist_ok=False)
run_dir.mkdir(parents=True, exist_ok=False)

manifest = run_rolling_backtest(
    pair_panel_path=str(PAIR_PANEL),
    canonical_pair_feature_path=str(CANONICAL_PAIR_FEATURE),
    selected_feature_path=str(SELECTED),
    model_contract_path=str(MODEL_CONTRACT),
    rolling_contract_path=str(ROLLING_CONTRACT),
    output_dir=str(report_dir),
    run_id=run_id,
    work9_root=str(WORK9),
)

print(json.dumps({
    'run_id': manifest['run_id'],
    'status': manifest['status'],
    'n_origins': manifest['n_origins'],
    'feature_parity': manifest['feature_parity'],
    'evaluation_rows': manifest['evaluation_rows'],
    'cumulative_3m_coverage': manifest['cumulative_3m_coverage'],
    'revision_evaluable': manifest['revision_evaluable'],
    'revision_rows': manifest['revision_rows'],
    'safety': manifest['safety'],
}, indent=2))


[ROLLING] origin=2025-01-01 fit=243,649 cal=53,475 eval=22,201


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/loc

[ROLLING] origin=2025-02-01 fit=288,597 cal=58,357 eval=23,690


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/loc

[ROLLING] origin=2025-03-01 fit=335,649 cal=62,921 eval=26,773


...

[ROLLING] origin=2025-11-01 fit=796,793 cal=118,438 eval=43,300


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/loc

[ROLLING] origin=2025-12-01 fit=867,380 cal=123,205 eval=44,289


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/loc

{
  "run_id": "rolling_backtest_v01_20260815T132319Z",
  "status": "PASS",
  "n_origins": 12,
  "feature_parity": {
    "parity_origin": "2025-12-01",
    "rolling_rows": 78864,
    "canonical_rows": 78864,
    "pass": true,
    "mismatch_columns": []
  },
  "evaluation_rows": 420846,
  "cumulative_3m_coverage": {
    "pair_origin_total": 142815,
    "pair_origin_complete_h1_h2_h3": 135448,
    "pair_origin_incomplete": 7367,
    "complete_rate": 0.9484157826558834
  },
  "revision_evaluable": true,
  "revision_rows": 2007256,
  "safety": {
    "supabase_accessed": false,
    "frozen_test_touched": false,
    "reconciliation_run": false,
    "model_freeze_run": false,
    "production_published": false,
    "current_status_used_as_predictor": false,
    "evaluation_labels_used_for_fit_or_calibration": false
  }
}


## 4. Audit evidence before accepting pointer

The main decision tables are model stability across origins, monthly/H1-H3 accuracy, cumulative 3M accuracy, and forecast revision.


In [6]:
stability = pd.read_csv(report_dir / 'model_stability_summary.csv')
display(stability)

score = pd.read_csv(report_dir / 'rolling_origin_scoreboard.csv')
overall = score[score['horizon'].isna() & score['segment'].isna()].sort_values('wape')
display(overall)

cum3 = pd.read_csv(report_dir / 'rolling_cumulative_3m_scoreboard.csv')
display(cum3.sort_values(['level', 'wape_3m']).head(80))

revision = pd.read_csv(report_dir / 'forecast_revision_scoreboard.csv')
display(revision.sort_values(['level', 'transition', 'revision_ratio_vs_old_forecast']).head(80))

gate = pd.read_csv(report_dir / 'rolling_behavior_gate.csv')
display(gate.groupby(['horizon', 'behavior_segment', 'chosen_expert']).size().rename('origin_count').reset_index())


,model,n_origins,mean_origin_wape,median_origin_wape,std_origin_wape,mean_abs_bias_ratio,max_abs_bias_ratio,underforecast_origin_rate,origin_wins,pair_wape_3m,base_sku_wape_3m
0,behavior_gated,12,0.912868,0.920406,0.032967,0.600140,0.770851,1.000000,2,0.790624,0.651906
1,hurdle,12,0.914394,0.920729,0.033712,0.619944,0.870612,1.000000,10,0.796985,0.667497
2,lightgbm_tweedie,12,1.011860,1.000365,0.058278,0.195626,0.393351,0.916667,0,0.777706,0.447575
3,moving_average_3,12,1.179773,1.198790,0.119443,0.231464,0.521712,0.333333,0,0.947747,0.521809
4,naive_1,12,1.261904,1.224474,0.147904,0.271284,0.557357,0.416667,0,1.098953,0.579408
5,seasonal_naive_12,12,1.484327,1.546190,0.182463,0.248171,0.834166,0.750000,0,1.453273,1.193577
6,tsb,12,1.915957,1.933255,0.310375,1.129119,1.736656,0.000000,0,1.671900,1.244453
7,croston_sba,12,2.254266,2.305696,0.485625,1.460854,2.175290,0.000000,0,2.019531,1.584064


,model,universe,horizon,segment,n_rows,wape,mae,bias,bias_ratio,zero_false_positive_rate,positive_wape
49,behavior_gated,ROLLING_PRIMARY,NaN,NaN,420846,0.912378,37.661691,-1.048208e+07,-0.603392,0.055438,0.843327
42,hurdle,ROLLING_PRIMARY,NaN,NaN,420846,0.913978,37.727721,-1.084979e+07,-0.624558,0.049760,0.851072
35,lightgbm_tweedie,ROLLING_PRIMARY,NaN,NaN,420846,1.009720,41.679839,-3.334528e+06,-0.191949,1.000000,0.729106
14,moving_average_3,ROLLING_PRIMARY,NaN,NaN,420846,1.174415,48.478185,2.775047e+06,0.159743,0.509548,0.787638
0,naive_1,ROLLING_PRIMARY,NaN,NaN,420846,1.257152,51.893458,2.869591e+06,0.165185,0.259400,0.912816
7,seasonal_naive_12,ROLLING_PRIMARY,NaN,NaN,420846,1.482852,61.210031,-3.584260e+06,-0.206325,0.241318,1.053545
28,tsb,ROLLING_PRIMARY,NaN,NaN,420846,1.897477,78.325174,1.926043e+07,1.108710,0.994928,0.836280
21,croston_sba,ROLLING_PRIMARY,NaN,NaN,420846,2.225666,91.872346,2.484263e+07,1.430044,0.994928,0.834278


,model,level,n_units,wape_3m,mae_3m,bias_3m,bias_ratio_3m,actual_sum_m2,forecast_sum_m2
13,lightgbm_tweedie,BASE_SKU,12837,0.447575,5.770914e+02,-3.120186e+06,-0.188512,16551671.81,1.343149e+07
10,moving_average_3,BASE_SKU,12837,0.521809,6.728063e+02,2.733755e+06,0.165165,16551671.81,1.928543e+07
8,naive_1,BASE_SKU,12837,0.579408,7.470728e+02,2.680467e+06,0.161945,16551671.81,1.923214e+07
15,behavior_gated,BASE_SKU,12837,0.651906,8.405498e+02,-9.969170e+06,-0.602306,16551671.81,6.582502e+06
14,hurdle,BASE_SKU,12837,0.667497,8.606525e+02,-1.032847e+07,-0.624014,16551671.81,6.223205e+06
9,seasonal_naive_12,BASE_SKU,12837,1.193577,1.538965e+03,-3.114022e+06,-0.188139,16551671.81,1.343765e+07
12,tsb,BASE_SKU,12837,1.244453,1.604563e+03,1.885921e+07,1.139414,16551671.81,3.541088e+07
11,croston_sba,BASE_SKU,12837,1.584064,2.042448e+03,2.445585e+07,1.477546,16551671.81,4.100752e+07
21,lightgbm_tweedie,BRANCH,655,0.309998,7.833561e+03,-3.120186e+06,-0.188512,16551671.81,1.343149e+07
18,moving_average_3,BRANCH,655,0.355586,8.985573e+03,2.733755e+06,0.165165,16551671.81,1.928543e+07


,model,level,transition,n_units,revision_mae_m2,revision_ratio_vs_old_forecast,signed_revision_m2
11,croston_sba,BASE_SKU,H2_TO_H1,11625,30.641653,0.027950,-216632.937592
12,tsb,BASE_SKU,H2_TO_H1,11625,53.834569,0.055905,-517414.889370
10,moving_average_3,BASE_SKU,H2_TO_H1,11625,130.009694,0.243429,-187324.383333
13,lightgbm_tweedie,BASE_SKU,H2_TO_H1,11625,111.727631,0.303618,376769.438652
8,naive_1,BASE_SKU,H2_TO_H1,11625,289.152944,0.549208,-435073.550000
9,seasonal_naive_12,BASE_SKU,H2_TO_H1,11625,233.832458,0.628100,195361.920000
15,behavior_gated,BASE_SKU,H2_TO_H1,11625,147.802573,0.934667,842401.651607
14,hurdle,BASE_SKU,H2_TO_H1,11625,151.978465,1.042623,965747.449356
27,croston_sba,BASE_SKU,H3_TO_H2,11622,30.648858,0.028016,-216075.194426
28,tsb,BASE_SKU,H3_TO_H2,11622,53.799267,0.055999,-517911.129797


,horizon,behavior_segment,chosen_expert,origin_count
0,1,intermittent,hurdle,12
1,1,regular,hurdle,10
2,1,regular,lightgbm_tweedie,2
3,1,very_sparse,hurdle,12
4,2,intermittent,hurdle,12
5,2,regular,hurdle,10
6,2,regular,lightgbm_tweedie,2
7,2,very_sparse,hurdle,12
8,3,intermittent,hurdle,12
9,3,regular,hurdle,10


## 5. Accept rolling pointer only after PASS

This pointer records the backtest evidence only. It does **not** freeze a champion and does **not** authorize frozen test.


In [7]:
manifest_path = report_dir / 'backtest_manifest.json'
assert manifest['status'] == 'PASS'
assert manifest['n_origins'] >= 12
assert manifest['feature_parity']['pass'] is True
assert manifest['safety']['frozen_test_touched'] is False
assert manifest['safety']['evaluation_labels_used_for_fit_or_calibration'] is False
assert manifest['revision_evaluable'] is True, manifest.get('revision_rows')
assert manifest['revision_rows'] > 0

run_manifest = {
    'run_id': run_id,
    'run_type': 'ROLLING_ORIGIN_BACKTEST_V01',
    'status': 'PASS',
    'source_dataset_run_id': dataset_ptr['run_id'],
    'source_feature_run_id': feature_ptr['run_id'],
    'source_feature_selection_run_id': fs_ptr['run_id'],
    'source_model_candidate_run_id': model_ptr['run_id'],
    'backtest_manifest_path': str(manifest_path),
    'backtest_manifest_sha256': sha256_file(manifest_path),
    'frozen_test_touched': False,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
}
(run_dir / 'run_manifest.json').write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')

pointer = {
    'run_id': run_id,
    'status': 'PASS',
    'backtest_version': 'rolling_backtest_v01',
    'source_model_candidate_run_id': model_ptr['run_id'],
    'report_dir': str(report_dir),
    'run_manifest_path': str(run_dir / 'run_manifest.json'),
    'backtest_manifest_path': str(manifest_path),
    'backtest_manifest_sha256': sha256_file(manifest_path),
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
}
(CONFIG / 'current_rolling_backtest_run.json').write_text(json.dumps(pointer, indent=2), encoding='utf-8')

print('ROLLING BACKTEST PASS:', run_id)
print('Pointer:', CONFIG / 'current_rolling_backtest_run.json')
print('STOP HERE. Frozen test / reconciliation / freeze / production remain locked.')


ROLLING BACKTEST PASS: rolling_backtest_v01_20260815T132319Z
Pointer: /content/drive/MyDrive/work9/01_config/current_rolling_backtest_run.json
STOP HERE. Frozen test / reconciliation / freeze / production remain locked.
